# 03 — Potential release area

**Time:** 13:00–13:50

Typical (config name: *frequent*) is tighter. Infrequent (*extreme*) is more
inclusive, including start zones below 30°. We will keep the ISSW thresholds
for this session rather than retuning them to match the expert overlay.

This notebook runs PRA only (no Flow-Py).


In [ ]:
from pathlib import Path
import os, sys
sys.path.insert(0, str(Path(".").resolve()))
from workshop import REPO, DATA, OUT, AUTOATES_ROOT, need, read_raster, show
import matplotlib.pyplot as plt

os.chdir(REPO)
need(DATA / "01_elevation" / "inputs" / "alos_aw3d30_connaught.tif")
need(DATA / "02_forest" / "outputs_reference" / "sen2_canopy_cover.tif")
print("AUTOATES_ROOT", AUTOATES_ROOT)

In [ ]:
from autoates.comAutoATES.comAutoATES import load_config
from autoates.comAutoATES.in1.in1AutoATES import align_and_clip_rasters
from autoates.comAutoATES.pra.praCore import run_pra

cfg = load_config(REPO / "config" / "autoATESCfg_workshop.ini")
cfg.set("Processing", "run_runout", "False")
cfg.set("Processing", "run_ates", "False")
cfg.set("Processing", "run_colorize", "False")

pra_dir = OUT / "PRA"
pra_dir.mkdir(parents=True, exist_ok=True)
processed = align_and_clip_rasters(
    dem_path=Path(cfg["Input"]["dem_path"]),
    forest_canopy_path=Path(cfg["Input"]["forest_canopy_path"]),
    output_dir=pra_dir,
    aoi_path=Path(cfg["General"]["aoi_path"]),
    output_basename=cfg["General"]["output_basename"],
)
dem_path, forest_path = processed[0]
print("aligned DEM", dem_path.name, "forest", forest_path.name)

for scen in ("frequent", "extreme"):
    run_pra(dem_path, forest_path, pra_dir, scenario=scen, config=cfg)
print("PRA done")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, scen, title in (
    (axs[0], "frequent", "Typical PRA"),
    (axs[1], "extreme", "Infrequent PRA"),
):
    p = pra_dir / f"pra_{scen}_pra_sieve.tif"
    if not p.exists():
        p = DATA / "03_pra" / "outputs_reference" / f"pra_{scen}_pra_sieve.tif"
    arr, *_ = read_raster(p)
    show(arr, title, "cividis", ax=ax)
plt.tight_layout()

Look at open slopes below 30°: that is why the infrequent layer exists.
Polygons are in `outputs/connaught/PRA/pra_*_poly_v1.gpkg`.

**If stuck:** copy `data/03_pra/outputs_reference/` into `outputs/connaught/PRA/`.
